## Importing necessary libraries

In [ ]:
import pandas as pd

## Checking for missing values

In [ ]:
ds=pd.read_csv('movie_data.csv')

In [ ]:
ds.isnull().sum()

,0
title,0
description,22
directed_by,22
written_by,457
produced_by,87
starring,105
cinematography,210
edited_by,258
release_date,32
country,39


## Dropping irrelevant attributes

In [ ]:
#columns not needed
unwanted_cols=['written_by','cinematography','edited_by']
ds.drop(columns=unwanted_cols,axis=1,inplace=True)


## Dealing with missing data

In [ ]:
ds = ds.fillna("No data")


In [ ]:
ds.isnull().sum()

,0
title,0
description,0
directed_by,0
produced_by,0
starring,0
release_date,0
country,0
language,0


In [ ]:
ds.iloc[8,5]='15 August 2025'

In [ ]:
print(ds['release_date'].head(10))

0            14 February 2025
1                 4 July 2025
2    29 January 2025(Vietnam)
3       16 March 2025(Málaga)
4                7 March 2025
5                 16 May 2025
6                4 April 2025
7               June 20, 2025
8              15 August 2025
9                        2025
Name: release_date, dtype: object


## Removing date format ambiguity

In [ ]:
ds['clean_release_date'] = (
    ds['release_date']
    .str.replace(",", " ", regex=False)
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)


In [ ]:
print(ds['clean_release_date'].head(10))

0    14 February 2025
1         4 July 2025
2     29 January 2025
3       16 March 2025
4        7 March 2025
5         16 May 2025
6        4 April 2025
7       June 20  2025
8      15 August 2025
9                2025
Name: clean_release_date, dtype: object


## Function for changing all dates to date month year format

In [ ]:
def reorder_date(date_str):
    # where date data not present mostly future titles declared
    if pd.isnull(date_str) or date_str == 'No data':
        return 'No date'
    parts=str(date_str).split()
    # if only a year present future movies 2026,2027 etc
    if len(parts)==1 and parts[0].isdigit():
        return f'1 January {parts[0]}'
    # if in month day year format
    if len(parts)==3 and not parts[0].isdigit():
        parts[0],parts[1]=parts[1],parts[0]
        reordered_str=" ".join(parts)
        return reordered_str
    return date_str

In [ ]:
ds['reordered_date']=ds['clean_release_date'].apply(reorder_date)

In [ ]:
ds['reordered_date'].head(10)

,reordered_date
0,14 February 2025
1,4 July 2025
2,29 January 2025
3,16 March 2025
4,7 March 2025
5,16 May 2025
6,4 April 2025
7,20 June 2025
8,15 August 2025
9,1 January 2025


## Function for changing date to a standard dd-mm-yyyy

In [ ]:
def parse_date(date_str):
    if date_str == 'No date':
        return 'No date'
    dt = pd.to_datetime(date_str, dayfirst=True, errors='coerce')
    if pd.notnull(dt):
        return dt.strftime('%d-%m-%Y')
    else:
        return 'No date'

In [ ]:
ds['parsed_date']=ds['reordered_date'].apply(parse_date)

In [ ]:
ds['parsed_date'].head(10)

,parsed_date
0,14-02-2025
1,04-07-2025
2,29-01-2025
3,16-03-2025
4,07-03-2025
5,16-05-2025
6,04-04-2025
7,20-06-2025
8,15-08-2025
9,01-01-2025


In [ ]:
ds.drop(columns=['release_date','clean_release_date','reordered_date'],inplace=True)

In [ ]:
ds=ds.rename(columns={'parsed_date':'release_date'})

In [ ]:
ds.columns

Index(['title', 'description', 'directed_by', 'produced_by', 'starring',
       'country', 'language', 'release_date'],
      dtype='object')

In [ ]:
ds.to_csv('movie_cleaned_data.csv',index=False)